# 🧠 Memoria degli LLM
## Lezione pratica con Ollama, ChromaDB e LangChain

---

### Perché la memoria è importante?

Un LLM, per sua natura, è **stateless**: non ricorda nulla tra una chiamata e l'altra.
Ogni volta che inviamo un messaggio, il modello vede **solo** ciò che gli passiamo in quel momento.

Per costruire applicazioni conversazionali (chatbot, assistenti, ecc.) dobbiamo gestire noi stessi la memoria, e possiamo farlo in due modi:

| Tipo | Dove vive | Quanto dura | Uso tipico |
|------|-----------|-------------|------------|
| **Short-term** | Nel prompt (lista messaggi) | Finché la sessione è attiva | Conversazioni interattive |
| **Long-term** | In un database (vettoriale) | Permanente (o finché non cancellata) | Ricordare utenti, fatti, storico |

```
┌──────────────────────────────────────────────────────────┐
│                    ARCHITETTURA MEMORIA                  │
│                                                          │
│  ┌─────────────┐   context    ┌──────────────────────┐  │
│  │  Short-term │─────────────▶│                      │  │
│  │   Memory    │              │    LLM (Ollama)      │  │
│  │ (in-memory) │              │                      │  │
│  └─────────────┘              └──────────────────────┘  │
│                                          ▲               │
│  ┌─────────────┐   retrieval             │               │
│  │  Long-term  │─────────────────────────┘               │
│  │   Memory    │                                         │
│  │ (ChromaDB)  │                                         │
│  └─────────────┘                                         │
└──────────────────────────────────────────────────────────┘
```

---

## 📦 Installazione dipendenze

In [1]:
# Installa le librerie necessarie
%pip install ollama gradio chromadb langchain langchain-ollama langchain-chroma --quiet


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Modelli da scaricare

# ollama pull llama3.2
# ollama pull nomic-embed-text

---

# 🟡 PARTE 1 — Memoria a Breve Termine (Short-Term Memory)

### Come funziona?

La memoria a breve termine si gestisce **mantenendo una lista di messaggi** (la *conversation history*) che viene passata al modello ad ogni turno.
Il modello non ricorda nulla da solo: siamo noi a dargli il contesto, inserendo nel prompt tutti i messaggi precedenti.

```
Turno 1:  [system]  →  modello risponde
Turno 2:  [system, user_1, assistant_1, user_2]  →  modello risponde
Turno 3:  [system, user_1, assistant_1, user_2, assistant_2, user_3]  →  modello risponde
```

> **Limite**: la lista cresce ad ogni turno e a un certo punto supera la **context window** del modello.
> Strategie comuni: sliding window, summarization, top-k messaggi.

---

### 1.1 — Setup: importazioni e configurazione

In [1]:
import ollama

# Modello da usare (assicurati che sia disponibile: ollama pull llama3.2)
MODEL = "llama3.2"

# Messaggio di sistema: definisce la personalità/contesto del chatbot
SYSTEM_PROMPT = (
    "Sei un assistente utile e conciso. "
    "Rispondi sempre in italiano. "
    "Ricorda tutto ciò che l'utente ti ha detto durante questa conversazione."
)

print(f"✅ Modello configurato: {MODEL}")

✅ Modello configurato: llama3.2


### 1.2 — La funzione chiave: `chat_with_memory`

Questa funzione gestisce tutta la logica della memoria a breve termine.
Lo **stato** è semplicemente una lista Python di dizionari `{role, content}`.

In [2]:
def chat_with_memory(
    user_message: str,
    history: list[dict],
    model: str = MODEL,
    system_prompt: str = SYSTEM_PROMPT,
    max_messages: int = 20,
) -> tuple[str, list[dict]]:
    """
    Invia un messaggio al modello mantenendo la storia della conversazione.

    Args:
        user_message : testo dell'utente
        history      : lista dei messaggi precedenti [{role, content}, ...]
        model        : nome del modello Ollama
        system_prompt: istruzioni di sistema
        max_messages : finestra massima di messaggi (sliding window)

    Returns:
        (risposta del modello, history aggiornata)
    """
    # 1. Aggiunge il messaggio dell'utente alla storia
    history.append({"role": "user", "content": user_message})

    # 2. Sliding window: mantiene solo gli ultimi N messaggi
    #    In questo modo evitiamo di saturare la context window
    windowed_history = history[-max_messages:]

    # 3. Costruisce i messaggi da inviare: system + history
    messages = [{"role": "system", "content": system_prompt}] + windowed_history

    # 4. Chiama il modello Ollama
    response = ollama.chat(model=model, messages=messages)
    assistant_reply = response["message"]["content"]

    # 5. Aggiunge la risposta del modello alla storia
    history.append({"role": "assistant", "content": assistant_reply})

    return assistant_reply, history


def reset_memory() -> list[dict]:
    """Restituisce una history vuota (nuova sessione)."""
    return []


print("✅ Funzioni definite")

✅ Funzioni definite


### 1.3 — Test manuale della memoria

Verifichiamo che il modello **ricordi** le informazioni fornite nei turni precedenti.

In [3]:
# Inizializza una nuova sessione
history = reset_memory()

# --- Turno 1: forniamo informazioni personali ---
reply, history = chat_with_memory(
    user_message="Ciao! Mi chiamo Marco e sono un ingegnere informatico.",
    history=history,
)
print(f"👤 Utente : Ciao! Io mi chiamo Marco e sono un ingegnere informatico.")
print(f"🤖 Modello: {reply}\n")

# --- Turno 2: chiediamo qualcosa di correlato ---
reply, history = chat_with_memory(
    user_message="Quali linguaggi di programmazione mi consigli di imparare?",
    history=history,
)
print(f"👤 Utente : Quali linguaggi di programmazione mi consigli di imparare?")
print(f"🤖 Modello: {reply}\n")

# --- Turno 3: verifichiamo che il modello ricordi il nome ---
reply, history = chat_with_memory(
    user_message="Ricordi come mi chiamo?",
    history=history,
)
print(f"👤 Utente : Ricordi come mi chiamo e qual è il mio lavoro?")
print(f"🤖 Modello: {reply}\n")

print(f"📋 Messaggi in memoria: {len(history)}")

👤 Utente : Ciao! Io mi chiamo Marco e sono un ingegnere informatico.
🤖 Modello: Ciao Marco! Sono felice di aiutarti. Come posso essere utile oggi? Hai un problema specifico con cui ti trovi a confrontare, o forse hai bisogno di aiuto per un progetto in corso?

👤 Utente : Quali linguaggi di programmazione mi consigli di imparare?
🤖 Modello: Ecco alcuni linguaggi di programmazione che potrebbero essere utili per un ingegnere informatico come te, Marco:

1. **Python**: una delle lingue più popolari e versatile, spesso utilizzata per progetti di machine learning, data science e web development.
2. **JavaScript**: fondamentale per lo sviluppo di applicazioni web e client-side, oltre a essere utilizzato in molti progetti di mobile e desktop.
3. **C++**: un linguaggio di base per lo sviluppo di applicazioni di sistema, giochi e programmi di alta prestazione.
4. **Java**: un linguaggio di programmazione versatile e stabile, spesso utilizzato per lo sviluppo di applicazioni enterprise e mobili.

In [9]:
[{"role": "system", "content": SYSTEM_PROMPT}] + history

[{'role': 'system',
  'content': "Sei un assistente utile e conciso. Rispondi sempre in italiano. Ricorda tutto ciò che l'utente ti ha detto durante questa conversazione."},
 {'role': 'user',
  'content': 'Ciao! Mi chiamo Marco e sono un ingegnere informatico.'},
 {'role': 'assistant',
  'content': 'Ciao Marco! Sono felice di aiutarti. Sono un assistente utente pronto a supportarti. Come posso aiutarti oggi?'},
 {'role': 'user',
  'content': 'Quali linguaggi di programmazione mi consigli di imparare?'},
 {'role': 'assistant',
  'content': 'Marco, come ingegnere informatico, ci sono molti linguaggi di programmazione che potresti trovare utili a seconda delle tue esigenze. Ecco alcuni consigli:\n\n*   **Python**: un linguaggio di programmazione versatile e facile da imparare, molto utilizzato in campo data science, machine learning e web development.\n*   **Java**: un linguaggio di programmazione obbligatorio in molti corsi di laurea, molto utilizzato per lo sviluppo di applicazioni mobi

In [4]:
# Ispezioniamo la storia completa
print("📜 Storia della conversazione:\n")
for i, msg in enumerate(history):
    role_icon = "👤" if msg["role"] == "user" else "🤖"
    print(f"[{i+1}] {role_icon} {msg['role'].upper()}: {msg['content'][:120]}..." 
          if len(msg['content']) > 120 else f"[{i+1}] {role_icon} {msg['role'].upper()}: {msg['content']}")
    print()

📜 Storia della conversazione:

[1] 👤 USER: Ciao! Mi chiamo Marco e sono un ingegnere informatico.

[2] 🤖 ASSISTANT: Ciao Marco! Sono felice di aiutarti. Come posso essere utile oggi? Hai un problema specifico con cui ti trovi a confront...

[3] 👤 USER: Quali linguaggi di programmazione mi consigli di imparare?

[4] 🤖 ASSISTANT: Ecco alcuni linguaggi di programmazione che potrebbero essere utili per un ingegnere informatico come te, Marco:

1. **P...

[5] 👤 USER: Ricordi come mi chiamo?

[6] 🤖 ASSISTANT: Mi ricordo che ti chiami Marco e che sei un ingegnere informatico.



### 1.4 — Interfaccia Gradio per la chat con memoria a breve termine

Costruiamo una **chat UI** con Gradio che usa la funzione `chat_with_memory`.
Lo stato della conversazione è gestito da `gr.State`, che persiste tra un'interazione e l'altra.

In [5]:
import gradio as gr
import ollama


# =====================================================
# WRAPPER GRADIO
# =====================================================

def gradio_chat(user_message, chatbot_history, session_history):
    # chatbot_history è ciò che noi vediamo sull'interfaccia
    # session history è la memoria che usiamo per il modello

    if chatbot_history is None:
        chatbot_history = []

    if session_history is None:
        session_history = []

    if not user_message.strip():
        return "", chatbot_history, session_history

    reply, session_history = chat_with_memory(
        user_message=user_message,
        history=session_history,
    )

    chatbot_history.append(
        {
            "role": "user",
            "content": user_message,
        }
    )

    chatbot_history.append(
        {
            "role": "assistant",
            "content": reply,
        }
    )

    return "", chatbot_history, session_history


def gradio_reset():
    return [], []


# =====================================================
# INTERFACCIA
# =====================================================

with gr.Blocks(
    title="🧠 Short-Term Memory Chat",
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown("""
    # 🧠 Chat con Memoria a Breve Termine

    Ogni turno viene passata al modello
    una finestra degli ultimi messaggi.

    Prova a dire il tuo nome e poi chiedi:
    "Qual è il mio nome?"
    """)

    session_state = gr.State([])

    chatbot = gr.Chatbot(
        label="Conversazione",
        height=500,
    )

    with gr.Row():

        msg_input = gr.Textbox(
            label="Il tuo messaggio",
            placeholder="Scrivi un messaggio...",
            scale=5,
        )

        send_btn = gr.Button(
            "Invia 📨",
            variant="primary",
            scale=1,
        )

    reset_btn = gr.Button(
        "🔄 Nuova conversazione",
        variant="secondary",
    )

    gr.Markdown(
        "> ℹ️ Memoria impostata a 20 messaggi (sliding window)"
    )

    send_btn.click(
        fn=gradio_chat,
        inputs=[
            msg_input,
            chatbot,
            session_state,
        ],
        outputs=[
            msg_input,
            chatbot,
            session_state,
        ],
    )

    msg_input.submit(
        fn=gradio_chat,
        inputs=[
            msg_input,
            chatbot,
            session_state,
        ],
        outputs=[
            msg_input,
            chatbot,
            session_state,
        ],
    )

    reset_btn.click(
        fn=gradio_reset,
        inputs=[],
        outputs=[
            chatbot,
            session_state,
        ],
    )

demo.launch()

C:\Users\A1627apulia\AppData\Local\Temp\ipykernel_19292\3956448608.py:52: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


---

# 🟢 PARTE 2 — Memoria a Lungo Termine (Long-Term Memory)

### Come funziona?

La memoria a lungo termine sfrutta un **database vettoriale** (ChromaDB) per salvare i fatti importanti in modo **persistente**.
Quando l'utente invia un messaggio, il sistema:

1. **Salva** il messaggio (o i fatti estratti) come embedding nel DB
2. **Cerca** i ricordi più rilevanti via similarity search
3. **Inietta** i ricordi trovati nel prompt come contesto aggiuntivo

```
┌──────────┐   save    ┌─────────────────────┐
│  Utente  │──────────▶│   ChromaDB           │
│          │           │  (embedding store)   │
│          │  retrieve │                      │
│          │◀──────────│  similarity search   │
└──────────┘           └─────────────────────┘
      │                          │
      │ query + retrieved memories
      ▼
┌──────────────┐
│  LLM (Ollama)│
└──────────────┘
```

> **Vantaggio rispetto alla short-term**: i ricordi **sopravvivono** al riavvio dell'applicazione.
> Non servono tutti i messaggi: solo quelli **rilevanti** alla query corrente.

---

### 2.1 — Setup: importazioni e inizializzazione ChromaDB + LangChain

In [6]:
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
import chromadb
import uuid
import os

# --- Configurazione ---
EMBED_MODEL = "nomic-embed-text"   # ollama pull nomic-embed-text
CHAT_MODEL  = "llama3.2"           # ollama pull llama3.2
CHROMA_DIR  = "./chroma_ltm_db"    # cartella persistente su disco
COLLECTION  = "long_term_memory"   # nome collection ChromaDB
TOP_K       = 3                    # quanti ricordi recuperare per query

# --- LLM Ollama ---
llm = OllamaLLM(model=CHAT_MODEL)

# --- Embeddings Ollama ---
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

# --- Vector Store (ChromaDB persistente) ---
vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

print(f"✅ LLM         : {CHAT_MODEL}")
print(f"✅ Embeddings  : {EMBED_MODEL}")
print(f"✅ ChromaDB    : {CHROMA_DIR}")
print(f"✅ Ricordi nel DB: {vectorstore._collection.count()}")

✅ LLM         : llama3.2
✅ Embeddings  : nomic-embed-text
✅ ChromaDB    : ./chroma_ltm_db
✅ Ricordi nel DB: 0


### 2.2 — Funzioni core della memoria a lungo termine

In [7]:
def save_memory(text: str, metadata: dict | None = None) -> str:
    """
    Salva un testo nel vector store come ricordo permanente.

    Args:
        text     : contenuto del ricordo da salvare
        metadata : dizionario opzionale con informazioni aggiuntive

    Returns:
        id univoco del documento salvato
    """
    doc_id = str(uuid.uuid4())
    doc = Document(
        page_content=text,
        metadata=metadata or {"source": "user_conversation"},
    )
    vectorstore.add_documents([doc], ids=[doc_id])
    return doc_id


def retrieve_memories(query: str, k: int = TOP_K) -> list[str]:
    """
    Recupera i ricordi più rilevanti alla query tramite similarity search.

    Args:
        query : testo di ricerca
        k     : numero massimo di ricordi da restituire

    Returns:
        lista di stringhe con i ricordi più simili
    """
    docs = vectorstore.similarity_search(query, k=k)
    return [doc.page_content for doc in docs]


def clear_all_memories():
    """Cancella tutti i ricordi dal vector store."""
    ids = vectorstore._collection.get()["ids"]
    if ids:
        vectorstore._collection.delete(ids=ids)
    print(f"🗑️  Cancellati {len(ids)} ricordi.")


def count_memories() -> int:
    """Restituisce il numero di ricordi salvati."""
    return vectorstore._collection.count()


print("✅ Funzioni memoria a lungo termine definite")

✅ Funzioni memoria a lungo termine definite


### 2.3 — Il prompt con i ricordi iniettati

Il prompt template include una sezione `{memories}` dove vengono inseriti i ricordi recuperati da ChromaDB.

In [8]:
LONG_TERM_PROMPT = PromptTemplate(
    input_variables=["memories", "question"],
    template="""\
Sei un assistente personale con memoria a lungo termine.
Rispondi sempre in italiano, in modo conciso e preciso.

Di seguito trovi i ricordi rilevanti recuperati dalla tua memoria a lungo termine:

--- RICORDI ---
{memories}
---------------

Domanda dell'utente: {question}

Risposta:""",
)

print("✅ Prompt template configurato")
print("\nStruttura del prompt:")
print(LONG_TERM_PROMPT.template)

✅ Prompt template configurato

Struttura del prompt:
Sei un assistente personale con memoria a lungo termine.
Rispondi sempre in italiano, in modo conciso e preciso.

Di seguito trovi i ricordi rilevanti recuperati dalla tua memoria a lungo termine:

--- RICORDI ---
{memories}
---------------

Domanda dell'utente: {question}

Risposta:


### 2.4 — Funzione principale: `ask_with_long_term_memory`

In [9]:
def ask_with_long_term_memory(
    question: str,
    auto_save: bool = True,
) -> dict:
    """
    Pone una domanda al modello, recuperando ricordi rilevanti da ChromaDB.

    Args:
        question  : domanda dell'utente
        auto_save : se True, salva automaticamente la domanda come ricordo

    Returns:
        dizionario con 'answer', 'memories_used', 'memory_id'
    """
    # 1. Recupera ricordi rilevanti
    memories = retrieve_memories(question)
    memories_text = "\n".join(f"- {m}" for m in memories) if memories else "Nessun ricordo disponibile."

    # 2. Costruisce il prompt con i ricordi
    prompt_text = LONG_TERM_PROMPT.format(
        memories=memories_text,
        question=question,
    )

    # 3. Chiama il modello
    answer = llm.invoke(prompt_text)

    # 4. Salva la domanda come nuovo ricordo (opzionale)
    memory_id = None
    if auto_save:
        memory_id = save_memory(
            text=f"L'utente ha detto/chiesto: {question}",
            metadata={"type": "user_query"},
        )

    return {
        "answer": answer,
        "memories_used": memories,
        "memory_id": memory_id,
    }


print("✅ Funzione ask_with_long_term_memory definita")

✅ Funzione ask_with_long_term_memory definita


### 2.5 — Test: salvataggio e recupero di ricordi

Simuliamo una sessione in cui l'utente fornisce informazioni su sé stesso, poi chiudiamo e riapriamo la sessione: i ricordi devono **persistere**.

In [10]:
# --- Sessione 1: salvataggio di fatti importanti ---
print("=" * 50)
print("📝 SESSIONE 1 — Salvataggio ricordi")
print("=" * 50)

facts = [
    "Mi chiamo Alessio e sono uno sviluppatore Python con 5 anni di esperienza.",
    "Il mio linguaggio preferito è Python, ma sto imparando Rust.",
    "Lavoro in una startup che si occupa di intelligenza artificiale a Milano.",
    "Ho un cane che si chiama Rex, un labrador di 3 anni.",
    "Il mio piatto preferito è la carbonara.",
]

for fact in facts:
    mid = save_memory(fact, metadata={"type": "personal_info"})
    print(f"  💾 Salvato [{mid[:8]}...]: {fact}")

print(f"\n📊 Totale ricordi nel DB: {count_memories()}")

📝 SESSIONE 1 — Salvataggio ricordi
  💾 Salvato [4fa825c5...]: Mi chiamo Alessio e sono uno sviluppatore Python con 5 anni di esperienza.
  💾 Salvato [1262fa8a...]: Il mio linguaggio preferito è Python, ma sto imparando Rust.
  💾 Salvato [56cb2304...]: Lavoro in una startup che si occupa di intelligenza artificiale a Milano.
  💾 Salvato [a294873f...]: Ho un cane che si chiama Rex, un labrador di 3 anni.
  💾 Salvato [95c57dbe...]: Il mio piatto preferito è la carbonara.

📊 Totale ricordi nel DB: 5


In [11]:
# --- Sessione 2: recupero dei ricordi ---
print("=" * 50)
print("🔍 SESSIONE 2 — Recupero e utilizzo dei ricordi")
print("=" * 50)

questions = [
    "Qual è il mio linguaggio di programmazione preferito?",
    "Come si chiama il mio cane?",
    "Dove lavoro e di cosa si occupa la mia azienda?",
]

for question in questions:
    print(f"\n👤 Domanda : {question}")
    result = ask_with_long_term_memory(question, auto_save=False)
    print(f"🤖 Risposta: {result['answer']}")
    print(f"🧠 Ricordi usati:")
    for m in result["memories_used"]:
        print(f"   • {m}")

🔍 SESSIONE 2 — Recupero e utilizzo dei ricordi

👤 Domanda : Qual è il mio linguaggio di programmazione preferito?
🤖 Risposta: Il tuo linguaggio di programmazione preferito è Python.
🧠 Ricordi usati:
   • Il mio linguaggio preferito è Python, ma sto imparando Rust.
   • Il mio piatto preferito è la carbonara.
   • Mi chiamo Alessio e sono uno sviluppatore Python con 5 anni di esperienza.

👤 Domanda : Come si chiama il mio cane?
🤖 Risposta: Il tuo cane si chiama Rex.
🧠 Ricordi usati:
   • Ho un cane che si chiama Rex, un labrador di 3 anni.
   • Il mio piatto preferito è la carbonara.
   • Mi chiamo Alessio e sono uno sviluppatore Python con 5 anni di esperienza.

👤 Domanda : Dove lavoro e di cosa si occupa la mia azienda?
🤖 Risposta: Lavori a Milano in una startup che si occupa di intelligenza artificiale.
🧠 Ricordi usati:
   • Lavoro in una startup che si occupa di intelligenza artificiale a Milano.
   • Mi chiamo Alessio e sono uno sviluppatore Python con 5 anni di esperienza.
   • Il

In [12]:
# Confronto: stessa domanda SENZA memoria
print("=" * 50)
print("⚖️  CONFRONTO: con vs senza memoria a lungo termine")
print("=" * 50)

test_question = "Cosa sai di me?"

# Senza memoria
no_memory_answer = llm.invoke(
    f"Sei un assistente. Rispondi in italiano. Domanda: {test_question}"
)
print(f"\n❌ SENZA memoria:")
print(f"   {no_memory_answer}")

# Con memoria
result = ask_with_long_term_memory(test_question, auto_save=False)
print(f"\n✅ CON memoria ({len(result['memories_used'])} ricordi recuperati):")
print(f"   {result['answer']}")

⚖️  CONFRONTO: con vs senza memoria a lungo termine

❌ SENZA memoria:
   Non so nulla di te personalmente. Non abbiamo stabilito alcun contatto previo o non hai condiviso informazioni personali. Potresti voler condividere alcune informazioni su di te e su cosa mi chiedi? Sarò felice di aiutarti!

✅ CON memoria (3 ricordi recuperati):
   Sai che il tuo piatto preferito è la carbonara e che hai un cane di nome Rex, un labrador di 3 anni. Inoltre, conosci la tua affinità per i linguaggi di programmazione Python e Rust.


### 2.6 — Interfaccia Gradio per la memoria a lungo termine

L'interfaccia ha due tab:
- **Chat**: conversazione con recupero automatico dei ricordi
- **Gestione Memoria**: salva nuovi ricordi, visualizza quelli esistenti, elimina tutto

In [ ]:
import gradio as gr


# ── helpers UI ────────────────────────────────────────────────────────────────

def ltm_chat(user_message: str, chat_history: list):
    """Handler della chat con LTM."""
    if not user_message.strip():
        return "", chat_history

    result = ask_with_long_term_memory(user_message, auto_save=True)
    answer = result["answer"]

    # Aggiunge info sui ricordi usati in fondo alla risposta
    if result["memories_used"]:
        mem_info = "\n\n---\n🧠 *Ricordi usati:* " + " | ".join(
            [f"`{m[:60]}…`" if len(m) > 60 else f"`{m}`" for m in result["memories_used"]]
        )
        answer += mem_info

    chat_history.append((user_message, answer))
    return "", chat_history


def ltm_save_fact(fact_text: str):
    """Salva un nuovo ricordo manuale."""
    if not fact_text.strip():
        return "⚠️ Inserisci un testo valido.", ltm_list_memories()
    mid = save_memory(fact_text.strip(), metadata={"type": "manual_fact"})
    return f"✅ Ricordo salvato con ID: {mid[:16]}…", ltm_list_memories()


def ltm_list_memories():
    """Restituisce una stringa con tutti i ricordi salvati."""
    data = vectorstore._collection.get()
    if not data["documents"]:
        return "📭 Nessun ricordo salvato."
    lines = []
    for i, (doc, meta) in enumerate(zip(data["documents"], data["metadatas"])):
        lines.append(f"[{i+1}] ({meta.get('type', 'n/a')}) {doc}")
    return "\n".join(lines)


def ltm_clear():
    """Cancella tutti i ricordi."""
    clear_all_memories()
    return "🗑️ Tutti i ricordi eliminati.", ""


# ── Gradio UI ─────────────────────────────────────────────────────────────────

with gr.Blocks(title="🧠 Long-Term Memory Chat", theme=gr.themes.Soft()) as demo_ltm:
    gr.Markdown(
        """
        # 🧠 Chat con Memoria a Lungo Termine
        I ricordi vengono **salvati su ChromaDB** e **persistono** tra le sessioni.
        Il modello usa solo i ricordi **più rilevanti** alla domanda corrente.
        """
    )

    with gr.Tab("💬 Chat"):
        chatbot_ltm = gr.Chatbot(label="Conversazione", height=420)
        with gr.Row():
            msg_ltm = gr.Textbox(
                placeholder="Scrivi un messaggio... (verrà salvato come ricordo)",
                label="Messaggio",
                scale=5,
            )
            send_ltm = gr.Button("Invia 📨", scale=1, variant="primary")

        gr.Markdown(
            "> ℹ️ Ogni messaggio inviato viene **automaticamente salvato** come ricordo permanente."
        )

        send_ltm.click(
            fn=ltm_chat,
            inputs=[msg_ltm, chatbot_ltm],
            outputs=[msg_ltm, chatbot_ltm],
        )
        msg_ltm.submit(
            fn=ltm_chat,
            inputs=[msg_ltm, chatbot_ltm],
            outputs=[msg_ltm, chatbot_ltm],
        )

    with gr.Tab("🗄️ Gestione Memoria"):
        gr.Markdown("### Aggiungi un ricordo manuale")
        with gr.Row():
            fact_input = gr.Textbox(
                placeholder="Es: Mi chiamo Luca e sono un designer.",
                label="Nuovo ricordo",
                scale=4,
            )
            save_btn = gr.Button("💾 Salva", scale=1, variant="primary")

        save_status = gr.Textbox(label="Stato", interactive=False)

        gr.Markdown("### Ricordi salvati nel DB")
        memories_display = gr.Textbox(
            label="Ricordi",
            lines=12,
            interactive=False,
            value=ltm_list_memories,
        )

        with gr.Row():
            refresh_btn = gr.Button("🔄 Aggiorna lista", variant="secondary")
            clear_btn   = gr.Button("🗑️ Elimina tutti i ricordi", variant="stop")

        save_btn.click(
            fn=ltm_save_fact,
            inputs=[fact_input],
            outputs=[save_status, memories_display],
        )
        refresh_btn.click(
            fn=ltm_list_memories,
            outputs=[memories_display],
        )
        clear_btn.click(
            fn=ltm_clear,
            outputs=[save_status, memories_display],
        )

demo_ltm.launch()

C:\Users\A1627apulia\AppData\Local\Temp\ipykernel_19292\1887000587.py:52: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="🧠 Long-Term Memory Chat", theme=gr.themes.Soft()) as demo_ltm:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


: 

---

# 📊 Riepilogo: Short-Term vs Long-Term Memory

| Caratteristica | Short-Term | Long-Term |
|---|---|---|
| **Dove vive** | Lista Python in memoria RAM | ChromaDB su disco |
| **Persiste al riavvio** | ❌ No | ✅ Sì |
| **Come funziona** | Tutti i messaggi nel prompt | Similarity search + inject |
| **Scalabilità** | Limitata dalla context window | Praticamente illimitata |
| **Librerie usate** | `ollama` nativo | `langchain` + `chromadb` |
| **Costo computazionale** | Basso | Medio (embedding a ogni turno) |
| **Uso ideale** | Chat interattive, sessioni brevi | Profili utente, knowledge base |

---

### 🔗 Approccio ibrido (best practice)

In produzione si usano **entrambi** i tipi insieme:

```
prompt = system_prompt
       + long_term_memories   ← ChromaDB (fatti persistenti)
       + short_term_history   ← ultimi N messaggi della sessione
       + current_user_message
```

Questo garantisce **contesto immediato** (short-term) e **ricordo persistente** (long-term).